In [ ]:
import panel as pn

pn.extension()

The `Overlay` layout floats one or more panels at anchored positions on top of a `base` component (e.g. a map, plot, image or video) *without* the panels consuming layout space or displacing the base. Unlike a full-size covering layer, each floating panel is only as large as its own footprint, so the base stays interactive (pan/zoom/click) everywhere else.

Each panel is declared as a `(where, obj)` tuple where `where` is either one of the nine named anchors of a 3x3 grid (`"top-left"`, `"top-center"`, `"top-right"`, `"center-left"`, `"center"`, `"center-right"`, `"bottom-left"`, `"bottom-center"`, `"bottom-right"`) or an explicit `(x, y)` coordinate -- in pixels or as a percentage string -- measured from the top-left corner of `base`. To place more than one thing at a single anchor, compose them first with `Row`, `Column` or `FlexBox`.

#### Parameters:

* **``base``** (Viewable): The underlay component the panels float on top of. Determines the size of the `Overlay` unless overridden by `sizing_mode`, `width` and/or `height`.
* **``objects``** (list): The list of floating panels, each placed at the anchor or coordinate it was declared with. Should not generally be modified directly except when replaced in its entirety -- use the list-like methods below (`append`, `insert`, etc.) instead.
* **``anchors``** (list, read-only): Each panel's anchor or coordinate, aligned with `objects`.

Styling related parameters:

* **``margin``** (int or tuple): Space around the *whole* `Overlay` in its parent layout. This does not inset the panels -- that is controlled by each panel's own `margin` instead (see below). Defaults to `0` so a full-bleed base has no stray outer gap.

For details on other options for customizing the component see the [layout](../../how_to/layout/index.md) and [styling](../../how_to/styling/index.md) how-to guides.

___

### Anchors

A classic use case -- inspired by tools like Breakthrough Energy's *Contrails Map* -- is pinning a legend, some controls, and a guide card around the corners of a full-bleed map or plot that stays interactive underneath. Here we stand in for the map with a plain colored box:

In [ ]:
def panel_box(text, color):
    return pn.pane.HTML(
        f'<div style="width:100%;height:100%;color:white;'
        f'display:flex;align-items:center;justify-content:center;">{text}</div>',
        styles={'background': color, 'border-radius': '4px'},
        width=120, height=40,
    )

base_map = pn.pane.HTML(
    '<div style="width:100%;height:100%;"></div>',
    styles={'background': '#2b2b2b'},
)

overlay = pn.layout.Overlay(
    ('top-left', panel_box('Legend', '#3b82f6')),
    ('top-right', pn.Column(panel_box('Search', '#10b981'), panel_box('Layers', '#10b981'))),
    ('bottom-center', panel_box('◀ ■ ▶', '#f59e0b')),
    ('bottom-right', panel_box('Guide', '#ef4444')),
    base=base_map,
    sizing_mode='stretch_both',
    height=320,
)
overlay

Since only `base` occupies layout space, the `Overlay` above is exactly as tall as the `height` we gave it, with the four panels floating on top and the dark background filling the rest -- try resizing the browser window and the panels stay pinned to their corners while the base keeps filling the available space.

The read-only `anchors` property stays aligned with `objects`, in declaration order:

In [ ]:
overlay.anchors

### Coordinates

Instead of a named anchor, `where` may also be an explicit `(x, y)` coordinate measured from the top-left of `base`. Values may be pixels (`int`) or percentages (`str`), and unlike named anchors, coordinates may repeat freely:

In [ ]:
chart = pn.pane.HTML(
    '<div style="width:100%;height:100%;"></div>',
    styles={'background': '#1f2937'}, width=400, height=250,
)

pn.layout.Overlay(
    ((24, 16), panel_box('Pixels: (24, 16)', '#8b5cf6')),
    (("60%", "70%"), panel_box('Percent: (60%, 70%)', '#ec4899')),
    base=chart,
)

### Sizing

Only `base` is in normal flow -- panels are absolutely positioned and never affect the `Overlay`'s size. That leaves two ways to size it:

* **Content-sized** (default, `sizing_mode=None`): the `Overlay` hugs `base`'s natural size, exactly like the fixed-size `chart` example above.
* **Responsive** (`sizing_mode="stretch_both"`, `"stretch_width"`, or an explicit `width`/`height`): the `Overlay` takes that size and `base` is stretched to fill it -- this is the full-bleed map/plot case used at the top of this page.

Mixing the two the wrong way round -- a responsively sized `Overlay` around a fixed-size `base` -- leaves the panels anchored to empty space beyond the smaller base, so pick whichever of the two matches how `base` itself is sized.

### Updating

Like `Row` and `Column`, `Overlay` has a list-like API -- `append`, `extend`, `clear`, `insert`, `pop`, `remove`, `reverse` and `__setitem__` -- for interactively updating the panels, keeping `anchors` aligned automatically:

In [ ]:
overlay.append(('center', panel_box('New!', '#06b6d4')))
overlay.anchors

In [ ]:
# Re-assigning an existing panel's tuple moves it to a new anchor
overlay[0] = ('bottom-left', overlay[0])
overlay.anchors

### Validation

Placement is always explicit: a bare object with no anchor, an unrecognized anchor name, or reusing the same named anchor twice all raise a `ValueError` rather than silently doing the wrong thing. Coordinates are exempt from the uniqueness check since they're free placement:

In [ ]:
try:
    pn.layout.Overlay(('top-left', 'A'), ('top-left', 'B'))
except ValueError as e:
    print(e)